# SAM3 파츠 세그먼트 + 마스크 경계 오버레이 검증 — 옷 전체 크롭 vs 파츠별 조각

### 이 노트북이 답하려는 질문
1. 옷 전체를 크롭+마스킹한 뒤 곧바로 하자를 탐지하는 현재 방식과, 옷 -> 파츠(소매/옷깃/밑단 등)로 한 번 더 조각내서 그 안에서 하자를 탐지하는 방식 중 어느 쪽이 더 정확한가(특히 미세 하자).
2. bbox 사각형 대신 탐지 마스크의 윤곽선 + 반투명 채우기로 하자 위치를 표시하면 실제 형태(찢어짐 등 불규칙한 모양)를 더 잘 보여주는가.

### 실행 순서
0. 의존성 설치 -> HF 로그인 -> GPU 확인 -> 설정 -> 모델 로드 -> 테스트 이미지
1. Step 1 — 의류 탐지 + 시각화
2. Step 2 — 파츠 탐지(의류 크롭 안에서) + 시각화
3. Step 3 — 파츠별 크롭 + 하자/양성 탐지
4. Step 4 — baseline(현재 방식) vs parts-based(새 방식) 후처리 결과 비교
5. Step 5 — 최종 오버레이 비교(bbox vs border, 파츠 경계 디버그)
6. 이미지 전체 배치 실행 + 결과 표

**프로덕션(`app/`, `worker/`)은 건드리지 않는다.** 여기서 검증되면 별도 계획 문서로 프로덕션 반영 여부를 결정한다.


## 0. 의존성 설치


In [ ]:
# sam3-condition-check 스파이크와 같은 의존성(torch/transformers/opencv)을 재사용한다.
%pip install -q -r "../sam3-condition-check/requirements-sam3.txt"


## 0-1. Hugging Face 로그인 (gated repo)


In [ ]:
import os

from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("HF_TOKEN 환경변수로 로그인했습니다.")
else:
    print("HF_TOKEN이 없습니다 — huggingface-cli login을 미리 실행했다면 그대로 진행됩니다.")


## 0-2. CUDA / GPU 확인


In [ ]:
import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


## 0-3. 경로·설정


In [ ]:
import sys

# 공유 유틸(qc_shared.py)은 이 폴더의 부모(validation/QUAL/)에 있다.
sys.path.insert(0, "..")
# 노트북 CWD는 이 폴더(sam3-parts-segment-check/) 자체라 로컬 모듈은 경로 추가 없이 바로 import된다.

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import qc_shared
from border_overlay import draw_bordered_findings, draw_part_boundaries
from parts import PART_CROP_PAD_RATIO, PART_PROMPTS_BY_GROUP, PART_SCORE_THRESHOLD, part_group_for
from sam3_caller import Sam3Caller, crop_and_mask_to

qc_shared.use_korean_font()  # 그래프 제목 한글이 두부(□□)로 깨지지 않게

# ── 실험용 변수 — 여기 값만 바꾸고 이 셀부터 재실행 ──────────────────────
CATEGORY = "TOP_KNIT"  # PART_PROMPTS_BY_GROUP 그룹(top/bottom)을 고르는 키
GARMENT_CROP_PAD_RATIO = 0.05


## 0-4. 모델 로드 — `Sam3Caller`


In [ ]:
caller = Sam3Caller()
caller._ensure_loaded()  # 로그·디바이스 확인용으로 미리 로드(첫 detect() 호출에서 자동으로도 됨)


## 0-5. 테스트 이미지 로드


In [ ]:
image_paths = qc_shared.list_images()
assert image_paths, "sample_images/ 에 이미지가 없습니다 (qc_shared.SHARED_IMAGE_DIR 참고)."
print(f"이미지 {len(image_paths)}장: {[p.name for p in image_paths]}")

# §1~§5는 이 기준 이미지 1장으로 단계별로 확인한다. §6에서 전체 배치를 돈다.
demo_path = image_paths[0]
demo_image = Image.open(demo_path).convert("RGB")

plt.figure(figsize=(5, 5))
plt.imshow(demo_image)
plt.title(f"기준 이미지: {demo_path.name}")
plt.axis("off")
plt.show()


## 1. Step 1 — 의류 탐지 + 시각화

`GARMENT_PROMPT`("clothing")로 의류 인스턴스를 찾고, 가장 큰 것을 채택한다(`qc_shared.pick_largest`). 마스크를 원본 위에 반투명으로 겹쳐 그려서 실제로 옷 전체를 잡았는지 확인한다.


In [ ]:
garment_threshold = qc_shared.BACKEND_THRESHOLDS["sam3"]["garment_score"]
ve_full = caller.vision_embeds(demo_image)
garment_instances = caller.detect(demo_image, "GARMENT", qc_shared.GARMENT_PROMPT, garment_threshold, ve_full)
garment = qc_shared.pick_largest(garment_instances)
print(f"의류 후보 {len(garment_instances)}건, 채택: {garment is not None}")

if garment is not None:
    garment_mask, garment_bbox = garment["mask"], garment["bbox"]
    overlay = np.array(demo_image).copy()
    overlay[garment_mask.astype(bool)] = (0.6 * overlay[garment_mask.astype(bool)] + 0.4 * np.array([0, 160, 255])).astype(
        np.uint8
    )
    plt.figure(figsize=(5, 5))
    plt.imshow(overlay)
    plt.title(f"의류 마스크 (score={garment['confidence']:.2f})")
    plt.axis("off")
    plt.show()
else:
    garment_mask, garment_bbox = None, None
    print("의류를 못 찾았습니다 — 아래 단계는 원본 전체로 폴백합니다.")


## 2. Step 2 — 파츠 탐지(의류 크롭 안에서) + 시각화

의류 크롭+마스킹 이미지 안에서 `PART_PROMPTS_BY_GROUP`의 파츠 프롬프트(옷깃/소매/몸판/밑단/주머니 등)를 하나씩 탐지한다. 파츠마다 다른 색 윤곽선으로 겹쳐 그려서 실제 파츠 경계와 맞는지 눈으로 확인한다(`draw_part_boundaries`).


In [ ]:
assert garment_mask is not None, "이 셀부터는 의류를 찾은 경우만 다룬다(§3 폴백 참고)."

garment_crop, garment_crop_box = crop_and_mask_to(demo_image, garment_bbox, garment_mask, GARMENT_CROP_PAD_RATIO)

part_prompts = PART_PROMPTS_BY_GROUP[part_group_for(CATEGORY)]
ve_garment = caller.vision_embeds(garment_crop)
part_instances = [
    inst
    for part_name, prompt in part_prompts.items()
    for inst in caller.detect(garment_crop, part_name, prompt, PART_SCORE_THRESHOLD, ve_garment)
]
print(f"탐지된 파츠: {[p['type'] for p in part_instances]}")

if part_instances:
    part_debug = draw_part_boundaries(garment_crop, part_instances, qc_shared.KOREAN_LABELS, qc_shared.korean_font_path)
    plt.figure(figsize=(5, 6))
    plt.imshow(part_debug)
    plt.title("파츠 경계 (색상별 파츠)")
    plt.axis("off")
    plt.show()
else:
    print("파츠를 하나도 못 찾았습니다 — 프롬프트 어휘를 조정해야 할 수 있습니다(parts.py).")


## 3. Step 3 — 파츠별 크롭 + 하자/양성 탐지

탐지된 파츠마다 그 파츠 영역만 다시 크롭+마스킹해서, 그 좁은 영역 안에서 하자·양성 프롬프트를 돈다. 파츠 하나를 예시로 크롭 이미지와 그 안의 탐지 결과를 확대해서 보여준다 — 전체 이미지에서는 작아서 뭉개지던 하자가 좁은 크롭에서는 더 뚜렷하게 잡히는지가 이 스파이크의 핵심 가설이다.


In [ ]:
defect_threshold = qc_shared.BACKEND_THRESHOLDS["sam3"]["score"]


def detect_defects_and_benigns(image):
    ve = caller.vision_embeds(image)
    defects = [
        inst
        for defect_type, prompt in qc_shared.DEFECT_PROMPTS.items()
        for inst in caller.detect(image, defect_type, prompt, defect_threshold, ve)
    ]
    benigns = [
        inst
        for benign_type, prompt in qc_shared.BENIGN_PROMPTS.items()
        for inst in caller.detect(image, benign_type, prompt, defect_threshold, ve)
    ]
    return defects, benigns


per_part_results = []
for part in part_instances:
    part_crop, part_crop_box = crop_and_mask_to(garment_crop, part["bbox"], part["mask"], PART_CROP_PAD_RATIO)
    defects, benigns = detect_defects_and_benigns(part_crop)
    per_part_results.append(
        {"part": part, "crop": part_crop, "crop_box": part_crop_box, "defects": defects, "benigns": benigns}
    )
    print(f"{part['type']}: 하자 후보 {len(defects)}건, 양성 후보 {len(benigns)}건")


In [ ]:
# 예시로 하자가 가장 많이 잡힌 파츠 1개를 확대해서 보여준다.
if per_part_results:
    example = max(per_part_results, key=lambda row: len(row["defects"]))
    example_overlay = draw_bordered_findings(
        example["crop"], example["defects"], qc_shared.KOREAN_LABELS, qc_shared.korean_font_path
    )
    plt.figure(figsize=(5, 5))
    plt.imshow(example_overlay)
    plt.title(f"예시 파츠: {example['part']['type']} (크롭 안 하자 탐지)")
    plt.axis("off")
    plt.show()
else:
    print("파츠가 없어 예시를 건너뜁니다.")


## 4. Step 4 — baseline(현재 방식) vs parts-based(새 방식) 후처리 결과 비교

두 파이프라인 모두 같은 공통 후처리(의류 겹침 필터 -> 양성 억제 -> cross-type dedup -> 번호매기기, `qc_shared`)를 태운다 — 차이는 오직 하자를 "어느 단계에서 모았는가"뿐이어야 공정한 비교다.


In [ ]:
def finalize(defects, benigns, mask):
    kept, _dropped_off_garment = qc_shared.filter_by_garment(defects, mask)
    kept, _dropped_benign = qc_shared.suppress_benign(kept, benigns)
    kept, _dropped_dupe = qc_shared.dedupe_cross_type(kept)
    return qc_shared.renumber(kept)


# --- baseline: 의류 크롭 전체에서 곧바로 탐지 ---
baseline_defects, baseline_benigns = detect_defects_and_benigns(garment_crop)
baseline_defects = qc_shared.remap_instances_to_full(baseline_defects, garment_crop_box, demo_image.size)
baseline_benigns = qc_shared.remap_instances_to_full(baseline_benigns, garment_crop_box, demo_image.size)
baseline_kept = finalize(baseline_defects, baseline_benigns, garment_mask)

# --- parts-based: 파츠별 결과를 모아서 의류 크롭 좌표 -> 원본 좌표로 remap ---
parts_defects, parts_benigns = [], []
for row in per_part_results:
    parts_defects.extend(qc_shared.remap_instances_to_full(row["defects"], row["crop_box"], garment_crop.size))
    parts_benigns.extend(qc_shared.remap_instances_to_full(row["benigns"], row["crop_box"], garment_crop.size))
parts_defects = qc_shared.remap_instances_to_full(parts_defects, garment_crop_box, demo_image.size)
parts_benigns = qc_shared.remap_instances_to_full(parts_benigns, garment_crop_box, demo_image.size)
parts_kept = finalize(parts_defects, parts_benigns, garment_mask)

print(f"baseline: {len(baseline_kept)}건 / parts-based: {len(parts_kept)}건")


## 5. Step 5 — 최종 오버레이 비교(bbox vs border)

왼쪽은 기존 방식(사각형 bbox, `qc_shared.draw_numbered_boxes`), 오른쪽은 새 방식(마스크 윤곽선+반투명 채우기, `draw_bordered_findings`)이다. 같은 이미지·같은 판정 로직 기준으로 표현 방식만 다르다.


In [ ]:
baseline_overlay = qc_shared.draw_numbered_boxes(demo_image, baseline_kept)
parts_overlay = draw_bordered_findings(demo_image, parts_kept, qc_shared.KOREAN_LABELS, qc_shared.korean_font_path)

fig, axes = plt.subplots(1, 2, figsize=(11, 6))
axes[0].imshow(baseline_overlay)
axes[0].set_title(f"baseline — bbox ({len(baseline_kept)}건)")
axes[0].axis("off")
axes[1].imshow(parts_overlay)
axes[1].set_title(f"parts-based — border ({len(parts_kept)}건)")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## 6. 이미지 전체 배치 실행 + 결과 표

§1~§5를 이미지마다 반복해서 건수만 표로 모은다(오버레이는 이미지가 많으면 노트북이 무거워지니 표만). 개별 이미지를 다시 자세히 보고 싶으면 `demo_path`/`demo_image`를 바꿔서 §1부터 재실행한다.


In [ ]:
import pandas as pd


def run_one(image):
    ve = caller.vision_embeds(image)
    g_instances = caller.detect(image, "GARMENT", qc_shared.GARMENT_PROMPT, garment_threshold, ve)
    g = qc_shared.pick_largest(g_instances)
    if g is None:
        defects, benigns = detect_defects_and_benigns(image)
        kept = finalize(defects, benigns, None)
        return {"garment_found": False, "parts_found": [], "baseline": len(kept), "parts_based": len(kept)}

    g_mask, g_bbox = g["mask"], g["bbox"]
    crop, crop_box = crop_and_mask_to(image, g_bbox, g_mask, GARMENT_CROP_PAD_RATIO)

    b_defects, b_benigns = detect_defects_and_benigns(crop)
    b_defects = qc_shared.remap_instances_to_full(b_defects, crop_box, image.size)
    b_benigns = qc_shared.remap_instances_to_full(b_benigns, crop_box, image.size)
    b_kept = finalize(b_defects, b_benigns, g_mask)

    ve_crop = caller.vision_embeds(crop)
    parts = [
        inst
        for name, prompt in part_prompts.items()
        for inst in caller.detect(crop, name, prompt, PART_SCORE_THRESHOLD, ve_crop)
    ]
    p_defects, p_benigns = [], []
    for part in parts:
        p_crop, p_box = crop_and_mask_to(crop, part["bbox"], part["mask"], PART_CROP_PAD_RATIO)
        d, b = detect_defects_and_benigns(p_crop)
        p_defects.extend(qc_shared.remap_instances_to_full(d, p_box, crop.size))
        p_benigns.extend(qc_shared.remap_instances_to_full(b, p_box, crop.size))
    p_defects = qc_shared.remap_instances_to_full(p_defects, crop_box, image.size)
    p_benigns = qc_shared.remap_instances_to_full(p_benigns, crop_box, image.size)
    p_kept = finalize(p_defects, p_benigns, g_mask)

    return {
        "garment_found": True,
        "parts_found": [p["type"] for p in parts],
        "baseline": len(b_kept),
        "parts_based": len(p_kept),
    }


rows = []
for path in image_paths:
    image = Image.open(path).convert("RGB")
    result = run_one(image)
    result["image"] = path.name
    rows.append(result)

summary_df = pd.DataFrame(rows)[["image", "garment_found", "parts_found", "baseline", "parts_based"]]
summary_df


## 결론 (실행 후 채울 것)

- 파츠 기반 탐지가 baseline보다 하자를 더/덜 잡았는가:
- 파츠 프롬프트(`parts.py`)가 실제로 각 파츠 경계를 잘 잡았는가, 어휘 보완이 필요한가:
- border 오버레이가 bbox보다 실제로 더 보기 좋은가(불규칙한 하자 모양 표현):

### 후속 작업
- 채택한다면 `docs/plans/AI/`에 프로덕션 반영 설계 계획을 먼저 쓴다(CH006 선례). 여기 코드를 그대로 승격하지 않는다 — 프로덕션은 계획 문서 승인 후에만 손댄다.
